# Presidio through an Azure Government App Service Environment

This notebook calls existing analyzer and anonymizer REST services. It does **not** run local Presidio engines or download spaCy models. Run it from an Azure VM/kernel with private-network access to your ASE and an accessible managed identity.

The example uses synthetic data only. Do not display real input PII, matched values, access tokens, or raw error responses. Clear notebook outputs before saving or sharing. Detection is not a guarantee that all PII has been found.

## Prerequisites

1. The **kernel host** must resolve both ASE hostnames and reach them over HTTPS (port 443). VNet connectivity alone does not guarantee private DNS resolution. Keep TLS verification enabled; configure your organization's trusted CA bundle through `REQUESTS_CA_BUNDLE` if needed.
2. Enable the host's system-assigned managed identity, or attach a user-assigned identity. The kernel must expose an identity endpoint supported by `ManagedIdentityCredential`; not every hosted notebook environment does. Do not use the App Service's own identity as the caller unless that is where the kernel runs.
3. On **each** App Service, open **Authentication > Microsoft identity provider** and identify its Entra app registration. In that registration's **Expose an API** page, find the registered **Application ID URI**. Configure `ANALYZER_API_RESOURCE` and `ANONYMIZER_API_RESOURCE` below with those exact identifiers, such as `api://<application-client-id>`. They are not necessarily the service's HTTPS URL, and they are not the Azure Resource Manager audience. Do not guess them.
4. The App Service authentication configuration must accept tokens for that API and allow the calling managed identity under its tenant, client-application, and identity restrictions. If the API requires app-role assignment, have an administrator assign the appropriate role to the managed identity's service principal. Azure RBAC permissions such as Reader or AcrPull do not grant access to these APIs. Do not broadly allow every caller to bypass an authorization problem.
5. Use the Azure Government tenant/issuer configuration for these services. Managed identity obtains tokens through the host endpoint; this notebook does not contact a hardcoded commercial-cloud login authority.

This notebook does not change Azure resources or permissions. Unknown API resource identifiers and missing authorization must be resolved before live execution.

## Kernel setup

From a terminal on the kernel host, create a dedicated uv-managed environment (Python 3.10 or newer):

```text
uv venv .venv-ase
uv pip install --python .venv-ase requests azure-identity ipykernel
```

Select that environment as this notebook's kernel in VS Code. For a Jupyter server, register it using `.venv-ase\Scripts\python.exe -m ipykernel install --user --name presidio-ase` on Windows, or `.venv-ase/bin/python -m ipykernel install --user --name presidio-ase` on Linux. Install packages on the remote kernel host, not just your local computer.

References: [Presidio REST API](https://data-privacy-stack.github.io/presidio/api-docs/api-docs.html), [App Service Entra authentication](https://learn.microsoft.com/en-us/azure/app-service/configure-authentication-provider-aad), [ManagedIdentityCredential](https://learn.microsoft.com/en-us/python/api/azure-identity/azure.identity.managedidentitycredential).


In [ ]:
import math
import os
from urllib.parse import urlsplit

import requests
from azure.core.exceptions import AzureError
from azure.identity import ManagedIdentityCredential

# Supply the registered API resources, not guessed hostname-based audiences.
ANALYZER_API_RESOURCE = os.environ.get("ANALYZER_API_RESOURCE", "")
ANONYMIZER_API_RESOURCE = os.environ.get("ANONYMIZER_API_RESOURCE", "")
MANAGED_IDENTITY_CLIENT_ID = os.environ.get("MANAGED_IDENTITY_CLIENT_ID", "").strip()

SERVICES = {
    "analyzer": {
        "url": os.environ.get(
            "ANALYZER_BASE_URL",
            "https://presidio-analyzer.ase-jtl.appserviceenvironment.us",
        ).rstrip("/"),
        "resource": ANALYZER_API_RESOURCE.strip(),
    },
    "anonymizer": {
        "url": os.environ.get(
            "ANONYMIZER_BASE_URL",
            "https://presidio-anonymizer.ase-jtl.appserviceenvironment.us",
        ).rstrip("/"),
        "resource": ANONYMIZER_API_RESOURCE.strip(),
    },
}
TIMEOUT = (10, 60)  # Connect and read timeouts, in seconds.


## Validate configuration and initialize authentication

Set both API resource variables above (or their environment variables) before continuing. For a user-assigned identity, also set `MANAGED_IDENTITY_CLIENT_ID` to its **client ID**, not its object/principal ID. Leave it empty for the host's system-assigned identity.

Each service gets a token for its own API resource with the `/.default` scope. The credential handles token caching and refresh. There is no interactive-login or client-secret fallback.


In [ ]:
def validate_services() -> None:
    for name, service in SERVICES.items():
        try:
            parsed = urlsplit(service["url"])
            port = parsed.port
        except ValueError:
            raise ValueError(f"Invalid {name} HTTPS base URL.") from None
        if (
            parsed.scheme != "https"
            or not parsed.hostname
            or parsed.username is not None
            or parsed.password is not None
            or parsed.path
            or parsed.query
            or parsed.fragment
            or port not in (None, 443)
        ):
            raise ValueError(f"Configure a root HTTPS base URL for {name}.")
        resource = service["resource"]
        if not resource or resource.endswith("/.default"):
            raise ValueError(
                f"Set {name.upper()}_API_RESOURCE to the registered Application ID URI "
                "without /.default."
            )


validate_services()
credential = (
    ManagedIdentityCredential(client_id=MANAGED_IDENTITY_CLIENT_ID)
    if MANAGED_IDENTITY_CLIENT_ID
    else ManagedIdentityCredential()
)
session = requests.Session()
# Preserve Bearer authentication even if the host has a .netrc file.
session.auth = lambda request: request


def api_request(name: str, path: str, payload: dict | None = None) -> requests.Response:
    service = SERVICES[name]
    scope = service["resource"].rstrip("/") + "/.default"
    try:
        token = credential.get_token(scope)
    except (AzureError, ValueError):
        raise RuntimeError(
            f"Token acquisition failed for {name}; check the host managed identity, "
            "registered API resource, and required app-role assignments."
        ) from None
    try:
        response = session.request(
            "GET" if payload is None else "POST",
            service["url"] + path,
            headers={"Authorization": "Bearer " + token.token},
            json=payload,
            timeout=TIMEOUT,
            allow_redirects=False,
            verify=True,
        )
    except requests.exceptions.SSLError:
        raise RuntimeError(f"TLS validation failed for {name}; check the trusted CA chain.") from None
    except requests.exceptions.Timeout:
        raise RuntimeError(f"Request timed out for {name}; check connectivity and service health.") from None
    except requests.exceptions.RequestException:
        raise RuntimeError(f"Request failed for {name}; check private DNS and HTTPS connectivity.") from None
    if response.status_code != 200:
        raise RuntimeError(
            f"{name} returned HTTP {response.status_code}; consult the troubleshooting table. "
            "Response body omitted to protect sensitive data."
        )
    return response


def response_json(response: requests.Response, name: str):
    try:
        return response.json()
    except ValueError:
        raise RuntimeError(f"{name} did not return valid JSON; check the endpoint and image version.") from None


In [ ]:
for name in SERVICES:
    api_request(name, "/health")
    print(f"{name}: health endpoint returned HTTP 200")


## Detect a synthetic email address

Use a narrow entity filter for this deterministic example. The original string must remain unchanged between analysis and anonymization because detection offsets refer to that exact string. Only entity types, positions, and scores are displayed, not matched text. An empty result is valid and means this request found no matching entities; it does not establish that arbitrary text is safe.


In [ ]:
def validate_detections(data, text: str) -> list[dict]:
    if not isinstance(data, list):
        raise RuntimeError("Analyzer response must be a list of detections.")
    detections = []
    for item in data:
        if not isinstance(item, dict):
            raise RuntimeError("Analyzer returned an invalid detection object.")
        entity_type = item.get("entity_type")
        start, end, score = item.get("start"), item.get("end"), item.get("score")
        if (
            not isinstance(entity_type, str)
            or not entity_type
            or type(start) is not int
            or type(end) is not int
            or not 0 <= start < end <= len(text)
            or type(score) not in (int, float)
            or not 0 <= score <= 1
            or not math.isfinite(score)
        ):
            raise RuntimeError("Analyzer returned invalid detection fields or boundaries.")
        detections.append({"entity_type": entity_type, "start": start, "end": end, "score": score})
    return detections


synthetic_text = "Contact alice@example.com."
analysis_response = api_request(
    "analyzer",
    "/analyze",
    {"text": synthetic_text, "language": "en", "entities": ["EMAIL_ADDRESS"]},
)
analyzer_results = validate_detections(response_json(analysis_response, "analyzer"), synthetic_text)
for detection in analyzer_results:
    print(detection)
assert [(item["entity_type"], item["start"], item["end"]) for item in analyzer_results] == [
    ("EMAIL_ADDRESS", 8, 25)
], "Synthetic email detection did not match the expected entity and boundaries."


## Replace the detected entity

The REST API uses `anonymizers`, whereas the Python SDK uses `operators`. This example explicitly replaces every detected span with `<ANONYMIZED>`; it does not use hashing, encryption, or partial masking. Full replacement protects detected spans only; undiscovered PII would remain in the output.


In [ ]:
anonymization_response = api_request(
    "anonymizer",
    "/anonymize",
    {
        "text": synthetic_text,
        "analyzer_results": analyzer_results,
        "anonymizers": {"DEFAULT": {"type": "replace", "new_value": "<ANONYMIZED>"}},
    },
)
anonymized_result = response_json(anonymization_response, "anonymizer")
if (
    not isinstance(anonymized_result, dict)
    or not isinstance(anonymized_result.get("text"), str)
    or not isinstance(anonymized_result.get("items"), list)
):
    raise RuntimeError("Anonymizer returned an unexpected response shape.")
assert anonymized_result["text"] == "Contact <ANONYMIZED>.", "Synthetic replacement differed from the expected output."
print(anonymized_result["text"])


In [ ]:
session.close()
credential.close()


## Troubleshooting and live acceptance

| Symptom | Check |
| --- | --- |
| Missing API resource | Obtain each registered Application ID URI; do not append `/.default` yourself. |
| DNS failure / connection timeout | Test resolution and HTTPS access from the actual kernel host; check ASE private DNS, routes, NSGs, and service availability. |
| TLS error | Fix certificate trust/hostname configuration. Never set `verify=False`. |
| Managed identity unavailable | Verify the execution host's identity endpoint is accessible to the kernel. For a user-assigned identity, check attachment and its client ID. |
| Token acquisition failure | Verify the registered resource exists in the host identity's tenant/cloud and any required app-role assignment. |
| HTTP 301/302 or another redirect | Verify the URL and App Service Authentication configuration. API calls should use access tokens, not follow an interactive login redirect. |
| HTTP 401 | Check token audience, accepted audiences, tenant/issuer, and token validity for the target service. Do not use an ARM token. |
| HTTP 403 | Check Easy Auth allowed client applications/identities/tenants, required API app roles, and App Service network access restrictions. |
| HTTP 400 / invalid JSON / unexpected shape | Check the deployed image version and the REST payload contract. Do not print raw response bodies containing input data. |
| Health failure / HTTP 5xx | Inspect service health and container startup. The container normally listens on port 3000; the notebook calls the App Service HTTPS endpoint, not `:3000` or local Docker port `:5002`. |
| Synthetic assertion failure | Check enabled recognizers, supported language `en`, and the deployed configuration. Do not weaken the assertion to conceal a failed example. |

For live acceptance, configure the two API resources, select the network-connected managed-identity kernel, and run all cells in order. Both health checks must return HTTP 200; the analyzer must report `EMAIL_ADDRESS` at positions 8 through 25 (exclusive end); the final synthetic output must be `Contact <ANONYMIZED>.`. Then close the clients and clear all outputs before saving. If a request cell fails after client initialization, run the cleanup cell before reinitializing the clients.

Offline notebook validation cannot establish your ASE connectivity or authorization. No real PII should be used for the initial test. This sample intentionally omits the original local notebook's registry mutation, custom TITLE/PRONOUN recognizers, and masking examples.
